In [1]:
!cd ../../nqs; python -m pip install -e .

Obtaining file:///scratch/malyshev/Research/NQS/nqs
  Preparing metadata (setup.py) ... done
  Attempting uninstall: nqs
    Found existing installation: nqs 0.1
    Uninstalling nqs-0.1:
      Successfully uninstalled nqs-0.1
  DEPRECATION: Legacy editable install of nqs==0.1 from file:///scratch/malyshev/Research/NQS/nqs (setup.py develop) is deprecated. pip 25.0 will enforce this behaviour change. A possible replacement is to add a pyproject.toml or enable --use-pep517, and use setuptools >= 64. If the resulting installation is not behaving as expected, try using --config-settings editable_mode=compat. Please consult the setuptools documentation for more information. Discussion can be found at https://github.com/pypa/pip/issues/11457
  Running setup.py develop for nqs


In [2]:
import os
os.environ['OMP_NUM_THREADS'] = "16" 
os.environ['MKL_NUM_THREADS'] = "16" 
# os.environ['CUBLAS_WORKSPACE_CONFIG'] = ":16:8" 
# os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

In [3]:
import os
import json

import numpy as np# Function to detect if the environment is Jupyter
def in_notebook():
    try:
        from IPython import get_ipython
        if 'IPKernelApp' not in get_ipython().config:  # Check if not notebook
            return False
    except Exception:
        return False
    return True


# Import the appropriate tqdm based on the environment
if in_notebook():
    from tqdm.notebook import tqdm as env_dependent_tqdm
else:
    from tqdm import tqdm as env_dependent_tqdm
import torch as pt
import pandas as pd

from matplotlib import pyplot as plt

In [4]:
# Function to detect if the environment is Jupyter
def in_notebook():
    try:
        from IPython import get_ipython
        if 'IPKernelApp' not in get_ipython().config:  # Check if not notebook
            return False
    except Exception:
        return False
    return True


# Import the appropriate tqdm based on the environment
if in_notebook():
    from tqdm.notebook import tqdm as env_dependent_tqdm
else:
    from tqdm import tqdm as env_dependent_tqdm

In [5]:
from nqs.base.hilbert_space import HilbertSpace

from nqs.applications.quantum_chemistry.molecule import GeometryConfig, MolConfig
from nqs.applications.quantum_chemistry.experiments.preparation import create_mol

from nqs.infrastructure.nested_data import Schedule
from nqs.applications.quantum_chemistry.experiments.calculations.sample import SamplingConfig
from nqs.applications.quantum_chemistry.experiments.energy_opt_exp import EnergyOptExpConfig, EnergyOptExp

from nqs.applications.quantum_chemistry import CHEMICAL_ACCURACY

In [6]:
%load_ext autoreload
%autoreload 2

In [7]:
import itertools

import os
import uuid
import shutil
import h5py

from openfermion.chem import MolecularData

from pyscf.pbc import gto as pbcgto
from pyscf.pbc import scf as pbcscf
from pyscf.pbc import df as pbcdf
from pyscf.pbc import cc as pbccc
from pyscf.pbc.tools import madelung as pbcmadelung

from openfermion import get_fermion_operator
from openfermion.transforms import jordan_wigner
from openfermion import get_sparse_operator

from scipy.sparse.linalg import eigsh as sparse_eigsh


from typing import List, Tuple


class PBCPySCFMolecularData(MolecularData):
    """A derived class from openfermion.hamiltonians.MolecularData. This class
    is created to store the PBC PySCF method objects as well as molecule data from
    a fixed basis set at a fixed geometry that is obtained from PySCF
    electronic structure packages. This class provides an interface to access
    the PySCF Hartree-Fock, MP, CI, Coupled-Cluster methods and their energies,
    density matrices and wavefunctions.
    Attributes:
        pyscf_data(dict): To store PySCF method objects temporarily.
    """
    def __init__(self,
                 geometry: List[Tuple[str, Tuple[float, float, float]]] = '',
                 basis: str = '',
                 multiplicity: int = 1,
                 charge: int = 0,
                 description: str = '',
                 filename: str = None,
                 data_directory: str = None,
                 unit: str = 'B',
                 lattice_vecs: List[Tuple[float, float, float]] = None,
                 dimension: int = None,
                 kpt_per_axes: Tuple[int, int, int] = (1, 1, 1),
                 df_method: str = 'GDF',
                 exxdiv: str = 'ewald',
                 max_fci_qubits: int = 20):
        super(PBCPySCFMolecularData, self).__init__(geometry,
                                                    basis,
                                                    multiplicity,
                                                    charge,
                                                    description,
                                                    filename,
                                                    data_directory)
        self.unit = unit
        self.lattice_vecs = lattice_vecs
        self.dimension = dimension
        self.pyscf_data = {}

        self.kpt_per_axes = kpt_per_axes
        self.kpt_num = np.asarray(self.kpt_per_axes).prod()
        self.kpts = self.pyscf_cell.make_kpts(self.kpt_per_axes)
        self.n_orbitals = self.pyscf_cell.nao_nr() * self.kpt_num
        self.n_qubits = 2 * self.n_orbitals

        assert exxdiv in (None, 'ewald')
        self.exxdiv = exxdiv
        self.madelung = pbcmadelung(self.pyscf_cell, self.kpts)
        self.madelung_corr = self.madelung * 0.5 * self.pyscf_cell.nelectron

        assert df_method in ('AFTDF', 'DF', 'GDF', 'MDF')
        self.df_method = df_method

        self.nuclear_repulsion = float(self.pyscf_cell.energy_nuc())
        #if self.exxdiv is None:
        self.nuclear_repulsion -= self.madelung_corr

        self.ccsd_t_energy = None

        self.fci_data = {}
        self.max_fci_qubits = max_fci_qubits

    @property
    def pyscf_cell(self):
        if self.pyscf_data.get('cell', None) is None:
            self.pyscf_data['cell'] = self.prepare_cell()

        return self.pyscf_data['cell']

    @property
    def pyscf_scf(self):
        if self.pyscf_data.get('scf', None) is None:
            self.pyscf_data['scf'] = self.compute_scf()

        return self.pyscf_data['scf']

    @property
    def pyscf_cc(self):
        if self.pyscf_data.get('cc', None) is None:
            self.pyscf_data['cc'] = pbccc.KRCCSD(self.pyscf_scf)

        return self.pyscf_data['cc']

    @property
    def pyscf_eris(self):
        if self.pyscf_data.get('eris', None) is None:
            self.pyscf_data['eris'] = self.pyscf_cc.ao2mo()

        return self.pyscf_data['eris']

    def get_n_alpha_electrons(self):
        """Return number of alpha electrons."""
        return super(PBCPySCFMolecularData, self).get_n_alpha_electrons() * self.kpt_num

    def get_n_beta_electrons(self):
        """Return number of beta electrons."""
        return super(PBCPySCFMolecularData, self).get_n_beta_electrons() * self.kpt_num

    def prepare_cell(self):
        cell = pbcgto.Cell()
        cell.atom = self.geometry
        cell.basis = self.basis

        cell.spin = self.multiplicity - 1
        cell.charge = self.charge
        cell.symmetry = False

        cell.unit = self.unit
        cell.a = self.lattice_vecs
        if self.dimension is not None:
            cell.dimension = self.dimension

        cell.build()
        self.pyscf_data['cell'] = cell

        return cell

    def compute_scf(self):
        if self.pyscf_cell.spin:
            pyscf_scf = pbcscf.KROHF(self.pyscf_cell)
        else:
            pyscf_scf = pbcscf.KRHF(self.pyscf_cell)

        df_func = getattr(pbcdf, self.df_method)
        pyscf_scf.with_df = df_func(self.pyscf_cell)

        pyscf_scf.kpts = self.kpts

        pyscf_scf.exxdiv = self.exxdiv
        pyscf_scf.verbose = 0
        pyscf_scf.run()

        # Hold pyscf data in molecule. They are required to compute density
        # matrices and other quantities.
        self.pyscf_data['scf'] = pyscf_scf
        self.canonical_orbitals = pyscf_scf.mo_coeff
        self.orbital_energies = pyscf_scf.mo_energy

        return pyscf_scf

    def compute_one_body_integrals(self):
        orb_per_kpt = self.pyscf_cell.nao_nr()
        one_body_integrals = np.zeros((self.n_orbitals, self.n_orbitals), dtype=np.complex128)
        core_hamiltonians = self.pyscf_scf.get_hcore()
        for kpt_idx in range(self.kpt_num):
            kpt_c_matrix = self.pyscf_scf.mo_coeff[kpt_idx] / np.sqrt(self.kpt_num)
            kpt_one_body_integrals = np.einsum('pi,pq,qj->ij',
                                               kpt_c_matrix.conj(),
                                               core_hamiltonians[kpt_idx],
                                               kpt_c_matrix)
            one_body_integrals[kpt_idx * orb_per_kpt:(kpt_idx + 1) * orb_per_kpt,
                               kpt_idx * orb_per_kpt:(kpt_idx + 1) * orb_per_kpt] = kpt_one_body_integrals

        return one_body_integrals

    def compute_two_body_integrals(self):
        orb_per_kpt = self.pyscf_cell.nao_nr()
        two_body_integrals = np.zeros([self.n_orbitals] * 4, dtype=np.complex128)
        for kpt_indices in itertools.product(range(self.kpt_num), range(self.kpt_num), range(self.kpt_num), range(self.kpt_num)):
            kpts = [self.kpts[kpt_idx] for kpt_idx in kpt_indices]
            kpt_c_matrices = [self.pyscf_scf.mo_coeff[kpt_idx] / np.sqrt(self.kpt_num) for kpt_idx in kpt_indices]
            cur_two_body_integrals = self.pyscf_scf.with_df.ao2mo(kpt_c_matrices,
                                                                  kpts,
                                                                  compact=False).reshape([orb_per_kpt] * 4)
            two_body_integrals[kpt_indices[0] * orb_per_kpt:(kpt_indices[0] + 1) * orb_per_kpt,
                               kpt_indices[1] * orb_per_kpt:(kpt_indices[1] + 1) * orb_per_kpt,
                               kpt_indices[2] * orb_per_kpt:(kpt_indices[2] + 1) * orb_per_kpt,
                               kpt_indices[3] * orb_per_kpt:(kpt_indices[3] + 1) * orb_per_kpt] = cur_two_body_integrals
        two_body_integrals = np.asarray(two_body_integrals.transpose((0, 2, 3, 1)), order='C')

        return two_body_integrals

    def compute_integrals(self):
        self.one_body_integrals = self.compute_one_body_integrals()
        self.two_body_integrals = self.compute_two_body_integrals()

        return self.one_body_integrals, self.two_body_integrals

    def compute_hf_energy(self):
        if self.hf_energy is None:
            self.hf_energy = float(self.pyscf_scf.e_tot)
            if self.exxdiv is None:
                self.hf_energy -= float(self.madelung_corr)

        return self.hf_energy

    def compute_ccsd_energy(self):
        if self.ccsd_energy is None:
            self.pyscf_cc.run(eris=self.pyscf_eris)
            self.ccsd_energy = self.pyscf_cc.e_tot
            if self.exxdiv is None:
                self.ccsd_energy -= self.madelung_corr

        return self.ccsd_energy

    def compute_ccsd_t_energy(self):
        if self.ccsd_t_energy is None:
            if self.ccsd_energy is None:
                self.ccsd_energy = self.compute_ccsd_energy()
            self.ccsd_t_energy = self.ccsd_energy + self.pyscf_cc.ccsd_t(eris=self.pyscf_eris)

        return self.ccsd_t_energy

    @property
    def molecular_ham(self):
        if self.fci_data.get('molecular_ham', None) is None:
            if (self.one_body_integrals is None) and (self.two_body_integrals is None):
                self.compute_integrals()
            self.fci_data['molecular_ham'] = self.get_molecular_hamiltonian()

        return self.fci_data['molecular_ham']

    @property
    def qubit_ham(self):
        if self.fci_data.get('qubit_ham', None) is None:
            self.fci_data['qubit_ham'] = jordan_wigner(get_fermion_operator(self.molecular_ham))

        return self.fci_data['qubit_ham']

    @property
    def sparse_ham(self):
        if self.fci_data.get('sparse_ham', None) is None:
            self.fci_data['sparse_ham'] = get_sparse_operator(self.qubit_ham)

        return self.fci_data['sparse_ham']
    #
    # @property
    # def fci_energy(self):
    #     if self.fci_data.get('energy', None) is None:
    #         self.run_fci()
    #
    #     return self.fci_data['energy']
    #
    # @fci_energy.setter
    # def fci_energy(self, value):
    #     if hasattr(self, 'fci_data'):
    #         self.fci_data['energy'] = value

    @property
    def fci_wf(self):
        if self.fci_data.get('wf', None) is None:
            self.run_fci()

        return self.fci_data['wf']

    def run_fci(self):
        assert self.fci_data.get('wf', None) is None
        assert self.fci_energy is None
        assert self.n_qubits <= self.max_fci_qubits
        v, w = sparse_eigsh(self.sparse_ham, which='SA')
        self.fci_energy = float(np.min(v))
        self.fci_data['wf'] = w[:, np.argmin(v)]

        return self.fci_energy, self.fci_wf

    def compute_fci_energy(self):
        if self.fci_energy is None:
            self.run_fci()

        return self.fci_energy

In [8]:
geometry = [["H", [0.0, 0.0, 1.0]],
["H", [0.0, 0.0, -1.0]],]
lattice_vecs = np.eye(3)*5.0
dimension = 1
kpt_per_axes = (4, 1, 1)

pyscf_cell =  PBCPySCFMolecularData(geometry=geometry,
                              basis='sto-3g',
                              description='hydrogen_attempt',
                              filename=None,
                              data_directory=None,
                              unit='B',
                              lattice_vecs=lattice_vecs,
                              dimension=dimension,
                              kpt_per_axes=kpt_per_axes,
                              df_method='GDF',
                              exxdiv='ewald',
                              max_fci_qubits=20)

In [9]:
pyscf_cell.compute_ccsd_energy()

E(RCCSD) = -1.086576319730904  E_corr = -0.03987820348700349


-1.0865763197309044

In [10]:
pyscf_cell.compute_fci_energy()


WARN: df_ao2mo: momentum conservation not found in the given k-points [[0.         0.         0.        ]
 [0.         0.         0.        ]
 [0.         0.         0.        ]
 [0.31415927 0.         0.        ]]


WARN: df_ao2mo: momentum conservation not found in the given k-points [[0.         0.         0.        ]
 [0.         0.         0.        ]
 [0.         0.         0.        ]
 [0.62831853 0.         0.        ]]


WARN: df_ao2mo: momentum conservation not found in the given k-points [[0.        0.        0.       ]
 [0.        0.        0.       ]
 [0.        0.        0.       ]
 [0.9424778 0.        0.       ]]


WARN: df_ao2mo: momentum conservation not found in the given k-points [[0.         0.         0.        ]
 [0.         0.         0.        ]
 [0.31415927 0.         0.        ]
 [0.         0.         0.        ]]


WARN: df_ao2mo: momentum conservation not found in the given k-points [[0.         0.         0.        ]
 [0.         0.         0.        ]
 

/scratch/malyshev/miniconda3/envs/anqs/lib/python3.12/site-packages/openfermion/chem/molecular_data.py:377: ComplexWarning: Casting complex values to real discards the imaginary part
  one_body_coefficients[2 * p, 2 * q] = one_body_integrals[p, q]
/scratch/malyshev/miniconda3/envs/anqs/lib/python3.12/site-packages/openfermion/chem/molecular_data.py:378: ComplexWarning: Casting complex values to real discards the imaginary part
  one_body_coefficients[2 * p + 1, 2 * q + 1] = one_body_integrals[p, q]
/scratch/malyshev/miniconda3/envs/anqs/lib/python3.12/site-packages/openfermion/chem/molecular_data.py:383: ComplexWarning: Casting complex values to real discards the imaginary part
  two_body_coefficients[2 * p, 2 * q + 1, 2 * r + 1, 2 * s] = two_body_integrals[
/scratch/malyshev/miniconda3/envs/anqs/lib/python3.12/site-packages/openfermion/chem/molecular_data.py:386: ComplexWarning: Casting complex values to real discards the imaginary part
  two_body_coefficients[2 * p + 1, 2 * q, 2 * r,

-1.0865791220324703

In [11]:
device = pt.device('cpu')
parent_dir = './cells'
rng_seed = 0

hs = HilbertSpace(qubit_num=pyscf_cell.n_qubits,
                  device=device,
                  parent_dir=parent_dir,
                  rng_seed=rng_seed,
                  popcount_mode='memory_efficient')

In [12]:
from nqs.stochastic.symmetries import ParticleNumberSymmetry, SpinHalfProjectionSymmetry, Z2Symmetry
from nqs.stochastic.maskers import LocallyDecomposableMasker

In [13]:
full_space_indices = pt.arange(2**hs.qubit_num).reshape(-1, 1).to(hs.device)
full_space_base_vecs = hs.base_idx2base_vec(full_space_indices)

In [14]:
symmetries = (
    ParticleNumberSymmetry(hilbert_space=hs,
                           particle_num=pyscf_cell.n_electrons * pyscf_cell.kpt_num),
    SpinHalfProjectionSymmetry(hilbert_space=hs,
                               spin=0)
)
masker = LocallyDecomposableMasker(hilbert_space=hs,
                                   symmetries=symmetries)
my_phys_mask = masker.mask(full_space_base_vecs)
print(my_phys_mask.sum())

tensor(4900)


/scratch/malyshev/Research/NQS/nqs/nqs/stochastic/maskers/locally_decomposable_masker.py:142: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.memo = pt.load(self.memo_fil

In [15]:
import openfermion
from openfermion.ops.representations import InteractionOperator
from openfermion.chem.molecular_data import spinorb_from_spatial
from openfermion.transforms import jordan_wigner

import pennylane as qml
from pennylane.pauli import PauliSentence, PauliWord
from pennylane.pauli.utils import _binary_matrix_from_pws
from pennylane.qchem.tapering import _reduced_row_echelon, _kernel
#from pennylane.operation import active_new_opmath

def compute_symmetry_generators(*,
                                pyscf_cell: PBCPySCFMolecularData = None,
                                qubit_ham: openfermion.ops.QubitOperator = None):
        n_qubits = pyscf_cell.n_qubits
        wires = qml.wires.Wires(tuple(range(n_qubits)))

        num_qubits = len(wires)

        # Generate binary matrix for qubit_op
        ps = PauliSentence()
        term_idx = 0
        for term, coeff in qubit_ham.terms.items():
            term_idx += 1
            pw = {}
            for qubit_idx, pauli_op in term:
                pw[qubit_idx] = pauli_op
            pw = PauliWord(pw)
            ps[pw] = coeff
        # ps = pauli_sentence(h)
        binary_matrix = _binary_matrix_from_pws(list(ps), num_qubits)

        # Get reduced row echelon form of binary matrix
        rref_binary_matrix = _reduced_row_echelon(binary_matrix)
        rref_binary_matrix_red = rref_binary_matrix[
            ~np.all(rref_binary_matrix == 0, axis=1)
        ]  # remove all-zero rows

        # Get kernel (i.e., nullspace) for trimmed binary matrix using gaussian elimination
        nullspace = _kernel(rref_binary_matrix_red)

        generators = []
        pauli_map = {"00": "I", "10": "X", "11": "Y", "01": "Z"}

        for null_vector in nullspace:
            tau = {}
            for idx, op in enumerate(zip(null_vector[:num_qubits], null_vector[num_qubits:])):
                x, z = op
                tau[idx] = pauli_map[f"{x}{z}"]

            ham = qml.pauli.PauliSentence({qml.pauli.PauliWord(tau): 1.0})
            ham = ham.operation(wires) #if active_new_opmath() else ham.hamiltonian(wires)
            generators.append(ham)

        return generators

def create_z2_symmetries(hs: HilbertSpace = None,
                         cell: PBCPySCFMolecularData = None,):
    hf_base_vec = ([0] * (2 * cell.pyscf_cell.nao_nr() - cell.n_electrons) + [1] * cell.n_electrons) * cell.kpt_num
    hf_base_vec = pt.tensor([hf_base_vec],
                            dtype=hs.idx_dtype,
                            device=hs.device)
    hf_base_vec = hf_base_vec[..., hs.inv_perm]

    generators = compute_symmetry_generators(pyscf_cell=cell,
                            qubit_ham=cell.qubit_ham)

    z2_symmetries = []
    for gen in generators:
        pauli_z_positions = hs.qubit_num - pt.tensor(list(gen.ops[0].wires)[::-1],
                                                     dtype=hs.idx_dtype,
                                                     device=hs.device) - 1

        pauli_z_positions = hs.perm[pauli_z_positions]
        z2_symmetry = Z2Symmetry(hilbert_space=hs,
                                 pauli_z_positions=pauli_z_positions,
                                 value=None)
        z2_symmetry.value = z2_symmetry.compute_acc_eig(hf_base_vec)
        z2_symmetries.append(z2_symmetry)

    return tuple(z2_symmetries)

In [16]:
z2_symmetries = create_z2_symmetries(hs=hs,
                      cell=pyscf_cell)

symmetries = (
    ParticleNumberSymmetry(hilbert_space=hs,
                           particle_num=pyscf_cell.n_electrons * pyscf_cell.kpt_num),
    SpinHalfProjectionSymmetry(hilbert_space=hs,
                               spin=0)
)
symmetries = symmetries + z2_symmetries
masker = LocallyDecomposableMasker(hilbert_space=hs,
                                   symmetries=symmetries)
my_phys_mask = masker.mask(full_space_base_vecs)
print(my_phys_mask.sum())

tensor(1252)


/scratch/malyshev/miniconda3/envs/anqs/lib/python3.12/site-packages/pennylane/ops/op_math/prod.py:499: PennyLaneDeprecationWarning: Prod.ops is deprecated and will be removed in future releases. You can access both (coeffs, ops) via op.terms() Also consider op.operands.
  warnings.warn(


In [17]:
%matplotlib widget

In [18]:
from nqs.stochastic.symmetries.abstract_locally_decomposable_symmetry import AbstractLocallyDecomposableSymmetry

In [19]:
from nqs.base.constants import BASE_INT_TYPE

class QuasimomSymmetry(AbstractLocallyDecomposableSymmetry):
    def __init__(self,
                  *args,
                  value: int = None,
                  kpt_num: int = None,
                  **kwargs):
        super(QuasimomSymmetry, self).__init__(*args, **kwargs)
        assert value is not None
        assert (0 <= value <= kpt_num - 1) or (value is None)
        self.value = value

        assert self.qubit_num % kpt_num == 0
        self.kpt_num = kpt_num
        self.orbitals_per_kpt = self.qubit_num // kpt_num
        self.orbital_quasimoms = pt.tile(pt.arange(self.kpt_num).reshape(-1, 1), (1, self.orbitals_per_kpt)).reshape(-1)
        self.orbital_quasimoms = self.orbital_quasimoms.to(self.device)
        self.max_acc_eigs = pt.remainder(pt.cumsum(self.orbital_quasimoms, dim=0), self.kpt_num)
        self.is_modular = True
        self.modulo = self.kpt_num

    @property
    def start_eig(self):
        return 0

    @property
    def spectrum_size(self):
        return self.kpt_num
    
    @property
    def ref_eig(self):
        return self.value
    
    def min_acc_eig(self, qubits_seen: int = None):
        return 0
    
    def max_acc_eig(self, qubits_seen: int = None):
        #return self.kpt_num - 1
        # #if qubits_seen
        # print(self.max_acc_eigs[:qubits_seen + 1].max(dim=0)[0])
        # #max_acc_eig = pt.remainder(pt.sum(self.orbital_quasimoms[:qubits_seen + 1]), self.kpt_num)
        if qubits_seen == 0:
            return 0
        else:
            return self.max_acc_eigs[:qubits_seen].max(dim=0)[0]
        #return max_acc_eig.cpu().item()
        #return self.kpt_num - 1
    
    @property
    def acc_eig2ordinal_mul_const(self):
        return 1

    @property
    def acc_eig2ordinal_add_const(self):
        return 0

    @property
    def acc_eig2ordinal_div_const(self):
        return 1
    
    def compute_part_eig(self,
                         qubit_idx: int = None,
                         base_vec: pt.Tensor = None) -> pt.Tensor:
        assert (0 <= qubit_idx) and (qubit_idx <= self.qubit_num)
        assert base_vec.dtype == BASE_INT_TYPE
        #print(self.orbital_quasimoms)
        #print(qubit_idx)
        #print(base_vec)
        return base_vec * self.orbital_quasimoms[qubit_idx]
    
    def update_acc_eig(self,
                       qubits_seen: int = None,
                       base_vec: pt.Tensor = None,
                       acc_eig: pt.Tensor = None) -> pt.Tensor:
        assert (0 <= qubits_seen) and (qubits_seen <= self.qubit_num)
        assert base_vec.dtype == BASE_INT_TYPE
        assert acc_eig.dtype == BASE_INT_TYPE
        #print(pt.remainder(acc_eig + self.compute_part_eig(qubits_seen, base_vec),
        #                    self.kpt_num))
        return pt.remainder(acc_eig + self.compute_part_eig(qubits_seen, base_vec),
                            self.kpt_num)
    

In [20]:
z2_symmetries = create_z2_symmetries(hs=hs,
                      cell=pyscf_cell)

symmetries = (
    ParticleNumberSymmetry(hilbert_space=hs,
                           particle_num=pyscf_cell.n_electrons * pyscf_cell.kpt_num),
    SpinHalfProjectionSymmetry(hilbert_space=hs,
                               spin=0)
)
symmetries = symmetries + z2_symmetries

quasimom_symmetry = QuasimomSymmetry(hilbert_space=hs,
                                     value=0,
                                     kpt_num=pyscf_cell.kpt_num)
symmetries = symmetries #+ (quasimom_symmetry,)
masker = LocallyDecomposableMasker(hilbert_space=hs,
                                   symmetries=symmetries)
my_phys_mask = masker.mask(full_space_base_vecs)
print(my_phys_mask.sum())

tensor(1252)


In [21]:
masker.memo.sum()

tensor(629)

In [22]:
from nqs.stochastic.observables import PauliObservable

In [23]:
ham = PauliObservable(hilbert_space=hs,
                      of_qubit_operator=pyscf_cell.qubit_ham)

In [24]:
from nqs.applications.quantum_chemistry.experiments.preparation import create_ansatz

In [25]:
mols_root_dir = parent_dir
mol_name = 'N2'
geom = [["N", [0.0, 0.0, -0.556]],
        ["N", [0.0, 0.0, 0.556]]]

geom_config = GeometryConfig(type='toy', idx=0)
geom_dir = os.path.join(mols_root_dir,
                       f'name={mol_name}',
                       'geometries',
                        geom_config.to_path_suffix())

if not os.path.exists(geom_dir):
    os.makedirs(geom_dir)

geom_filename = os.path.join(geom_dir, 'geom.json')

if not os.path.exists(geom_filename):
    with open(geom_filename, 'w') as f:
        json.dump(geom, f)
else:
    print(f'You are trying to overwrite an existing geometry at {geom_filename}')
mol_config = MolConfig(name=mol_name,
                       geom_config=geom_config,
                       basis='sto-3g')


You are trying to overwrite an existing geometry at ./cells/name=N2/geometries/type=toy/idx=0/geom.json


In [26]:
series_name = 'toy_example'
sampling_schedule = Schedule(schedule=(
    (0, SamplingConfig(sample_num=10**5, sample_indices=False)),
    (100, SamplingConfig(sample_num=10**6, sample_indices=False)),
    (200, SamplingConfig(sample_num=10**7, sample_indices=False)),
    (1000, SamplingConfig(sample_num=10**8, sample_indices=False)),
     
))
exp_config = EnergyOptExpConfig(mols_root_dir=parent_dir,
                                mol_config=mol_config,
                                series_name=series_name,
                                sampling_schedule=sampling_schedule)
exp_config.meta_ansatz_config.ansatz_type = 'LogPsiANQS'
exp_config.meta_ansatz_config.ansatz_config.dtype = pt.complex128
exp_config.meta_ansatz_config.ansatz_config.de_mode = 'NADE'
exp_config.meta_ansatz_config.ansatz_config.qubit_grouping_config.qubit_per_qudit = 1
exp_config.meta_ansatz_config.ansatz_config.local_sampling_config.masking_depth = 2
exp_config.meta_ansatz_config.ansatz_config.subtract_mean = False
exp_config.meta_ansatz_config.ansatz_config.main_subnet_config.use_res = False
exp_config.meta_ansatz_config.ansatz_config.main_subnet_config.bias_config.use_bias = False
exp_config.meta_ansatz_config.ansatz_config.main_subnet_config.activation_config.pattern_type = 'sanqs_paper'

# 

#exp_config.sampling_schedule = sampling_schedule

exp_config.loss_type = 'full_e_loc'
exp_config.local_energy_config.code_version = 'old'
exp_config.local_energy_config.use_tree_for_candidates = 'ham'

exp_config.proc_grad_schedule[0][1].use_sr = False
exp_config.proc_grad_schedule[0][1].sr_config.max_indices_num = 100

In [27]:
wf = create_ansatz(config=exp_config.meta_ansatz_config,
                       hs=hs,
                       masker=masker,
                       sign_structure=False)
wf = wf.to(hs.device)

In [28]:
indices, stats = wf.sample_stats(sample_num=1000)

In [29]:
indices, stats = wf.sample_indices_gumbel(sample_num=1000)

In [30]:
indices.shape

torch.Size([496, 1])

In [31]:
from nqs.applications.quantum_chemistry.experiments.preparation import create_opt
opt = create_opt(wf=wf, opt_config=exp_config.opt_schedule[0][1])

In [32]:
from nqs.applications.quantum_chemistry.experiments import bin_search_schedule
from nqs.applications.quantum_chemistry.experiments.calculations.sample import sample, SamplingResult
def wrapped_sample(iter_idx: int = None):
    sampling_config = bin_search_schedule(exp_config.sampling_schedule, iter_idx=iter_idx)
    sampling_result, _, _, _ = sample(wf=wf,
                                      config=sampling_config,
                                      starting_sample_num=sampling_config.sample_num)
    
    return sampling_result

In [33]:
from nqs.applications.quantum_chemistry.experiments.calculations.compute_local_energies import compute_local_energies
def wrapped_compute_loss(iter_idx: int = None,
                         sampling_result: SamplingResult = None):
    sampled_amps = wf.amplitude(sampling_result.indices)
    #print(sampled_amps)
    local_energy_result, _ = compute_local_energies(wf=wf,
                                                    sampling_result=sampling_result,
                                                    sampled_amps=sampled_amps.detach(),
                                                    ham=ham,
                                                    config=exp_config.local_energy_config,
                                                    sample_aware=False)
    loss_local_energies = local_energy_result.full_e_loc_mc_est.values - local_energy_result.full_e_loc_mc_est.mean
    loss_freqs = local_energy_result.full_e_loc_mc_est.freqs
    #print(loss_freqs)
    loss = 2 * (loss_freqs * pt.log(pt.conj(sampled_amps)) * loss_local_energies).sum().real

    return loss, local_energy_result.full_e_loc_mc_est.mean.real

In [34]:
sampling_result = wrapped_sample(iter_idx=0)

In [35]:
loss = wrapped_compute_loss(iter_idx=0, sampling_result=sampling_result)

In [36]:
iter_num = 1000

for iter_idx in (pbar := env_dependent_tqdm(range(iter_num))):
    opt.zero_grad()
    sampling_result = wrapped_sample(iter_idx=iter_idx)
    loss, energy = wrapped_compute_loss(iter_idx=iter_idx,
                                sampling_result=sampling_result)
    print(f'Iter {iter_idx}: Loss = {loss}, Energy = {energy}')
    loss.backward()
    wf.clip_grad_norm(1.0)
    opt.step()

  0%|          | 0/1000 [00:00<?, ?it/s]

Iter 0: Loss = 0.02815337255581473, Energy = -0.5037610831914952
Iter 1: Loss = -0.06609647606971389, Energy = -0.5902238000864469
Iter 2: Loss = -0.12179638882660007, Energy = -0.6614629352704848
Iter 3: Loss = -0.1532100400428187, Energy = -0.727017943062063
Iter 4: Loss = -0.13917858919703766, Energy = -0.7901118492021557
Iter 5: Loss = -0.12747510966770698, Energy = -0.8501805216920943
Iter 6: Loss = -0.15040143682707724, Energy = -0.9020423003315443
Iter 7: Loss = -0.14021399620519343, Energy = -0.9404850846718571
Iter 8: Loss = -0.12535570993690395, Energy = -0.9697178022948175
Iter 9: Loss = -0.12672856516703873, Energy = -0.9958226416080305
Iter 10: Loss = -0.12262784458280361, Energy = -1.016876257326504
Iter 11: Loss = -0.0992351029110126, Energy = -1.0315437930415534
Iter 12: Loss = -0.06536260328305438, Energy = -1.039522480674854
Iter 13: Loss = -0.04423599714458026, Energy = -1.0440691324439504
Iter 14: Loss = -0.03213954518915353, Energy = -1.0462272135176653
Iter 15: Lo

KeyboardInterrupt: 

In [138]:
iter_num = 1000

for iter_idx in (pbar := env_dependent_tqdm(range(iter_num))):
    opt.zero_grad()
    sampling_result = wrapped_sample(iter_idx=iter_idx)
    loss, energy = wrapped_compute_loss(iter_idx=iter_idx,
                                sampling_result=sampling_result)
    print(f'Iter {iter_idx}: Loss = {loss}, Energy = {energy}')
    loss.backward()
    wf.clip_grad_norm(1.0)
    opt.step()

  0%|          | 0/1000 [00:00<?, ?it/s]

Iter 0: Loss = 0.029794288446581434, Energy = -0.4980111448882135
Iter 1: Loss = -0.05583103953216101, Energy = -0.5982279321082993
Iter 2: Loss = -0.13608467302960386, Energy = -0.6885216088029831
Iter 3: Loss = -0.17537581534617366, Energy = -0.7608321233443747
Iter 4: Loss = -0.2113214833588733, Energy = -0.8329295332017352
Iter 5: Loss = -0.24285767804052122, Energy = -0.8975590533359058
Iter 6: Loss = -0.2496044502243177, Energy = -0.9462374639119686
Iter 7: Loss = -0.21472585085702858, Energy = -0.9825452298391023
Iter 8: Loss = -0.16118800380280043, Energy = -1.00823810396294
Iter 9: Loss = -0.12261664915192466, Energy = -1.026934329889208
Iter 10: Loss = -0.08012614863913965, Energy = -1.037644886932479
Iter 11: Loss = -0.043592832237303494, Energy = -1.0431892712538362
Iter 12: Loss = -0.02386385633997415, Energy = -1.0463985649903698
Iter 13: Loss = -0.01548186649827515, Energy = -1.0481214137988881
Iter 14: Loss = -0.007170938427576375, Energy = -1.0488901642932995
Iter 15: 

In [104]:
iter_num = 1000

for iter_idx in (pbar := env_dependent_tqdm(range(iter_num))):
    opt.zero_grad()
    sampling_result = wrapped_sample(iter_idx=iter_idx)
    loss, energy = wrapped_compute_loss(iter_idx=iter_idx,
                                sampling_result=sampling_result)
    print(f'Iter {iter_idx}: Loss = {loss}, Energy = {energy}')
    loss.backward()
    wf.clip_grad_norm(1.0)
    opt.step()



  0%|          | 0/1000 [00:00<?, ?it/s]

Iter 0: Loss = 0.048092533342645724, Energy = -0.47125719988355674
Iter 1: Loss = 0.04602775560042528, Energy = -0.5314475364639855
Iter 2: Loss = 0.05290979251086597, Energy = -0.564775091141047
Iter 3: Loss = 0.032400499525687576, Energy = -0.5991776994868889
Iter 4: Loss = 0.007207927115579847, Energy = -0.6314873068708375
Iter 5: Loss = -0.04664060357192722, Energy = -0.6694945300897643
Iter 6: Loss = -0.08525738583276611, Energy = -0.7059417071813715
Iter 7: Loss = -0.11053573687041379, Energy = -0.7395408616191272
Iter 8: Loss = -0.07873392817068928, Energy = -0.7653707007278263
Iter 9: Loss = -0.04644150439931411, Energy = -0.778968320438816
Iter 10: Loss = -0.025953288949667468, Energy = -0.7878012355330043
Iter 11: Loss = -0.007644834120197015, Energy = -0.7939235140168854
Iter 12: Loss = 0.0049549783541494455, Energy = -0.7980294635819938
Iter 13: Loss = 0.01575962027105912, Energy = -0.8047196571180768
Iter 14: Loss = 0.01904316194575273, Energy = -0.8104771824649961
Iter 15